In [1]:
import sys
sys.path.insert(0, '..')
from dependencies import *

envelopes_log = eelbrain.load.unpickle(PROCESSED_PREDICTOR_DIR / f'~processed_envelopes-log.pickle')
durations = get_durations(envelopes_log)
models = get_models()

In [3]:
# LOAD EEG DATA

# Create dictionaries to hold the eeg scans of each subject, and the predictors of each model
subject_eegs = {} 
subject_model_predictors = {}

for subject in SUBJECTS:
    # Extract the raw files
    raw = mne.io.read_raw(EEG_DIR / subject / f'{subject}_alice-raw.fif', preload=True)
    # Filter the raw data to the desired band
    raw.filter(LOW_FREQUENCY, HIGH_FREQUENCY, n_jobs=1)
    # Interpolate bad channels
    # raw.interpolate_bads() # <-- No bad filters so commented out

    # Load the events embeded in the raw file
    events = eelbrain.load.fiff.events(raw)

    # Not all subjects have all trials; determine which stimuli are present
    trial_indexes = [STIMULI.index(stimulus) for stimulus in events['event']]
    # Extract the EEG data segments corresponding to the stimuli
    trial_durations = [durations[i] for i in trial_indexes]
    # Load the EEG with corresponding epochs
    eeg = eelbrain.load.fiff.variable_length_epochs(events, -0.100, trial_durations, decim=5)
    # Since trials are of unequal length, we will concatenate them for the TRF estimation.
    eeg_concatenated = eelbrain.concatenate(eeg)

    # Save pickle
    eelbrain.save.pickle(eeg_concatenated, STIMULUS_DIR / f'{subject}concatenated_eeg.pickle')

    # Update dictionaries
    subject_eegs[subject] = eeg_concatenated
    subject_model_predictors[subject] = {}

    for model, predictors in models.items():
        # print(f"Concatenating: {subject} ~ {model} predictors")
        # Select and concetenate the predictors corresponding to the EEG trials
        predictors_concatenated = []
        for predictor in predictors:
            predictors_concatenated.append(eelbrain.concatenate([predictor[i] for i in trial_indexes]))
        subject_model_predictors[subject][model] = predictors_concatenated

    # Save pickle
    eelbrain.save.pickle(subject_model_predictors, PREDICTOR_DIR / f'{subject}concatenated_predictors.pickle')
    print(f'{subject} predictors saved')


S05 predictors saved
S34 predictors saved
S35 predictors saved
S03 predictors saved
S04 predictors saved
S44 predictors saved
S26 predictors saved
S19 predictors saved
S21 predictors saved
S17 predictors saved
S10 predictors saved
S42 predictors saved
S45 predictors saved
S11 predictors saved
S16 predictors saved
S20 predictors saved
S18 predictors saved
S01 predictors saved
S39 predictors saved
S06 predictors saved
S08 predictors saved
S37 predictors saved
S36 predictors saved
S38 predictors saved
S22 predictors saved
S25 predictors saved
S13 predictors saved
S14 predictors saved
S40 predictors saved
S15 predictors saved
S12 predictors saved
S48 predictors saved
S41 predictors saved
